In [1]:
import numpy as np
import matplotlib.pyplot as plt

import torch
from datasets import load_dataset
import pandas as pd

import datasets

from transformers import AutoTokenizer, AutoModel

In [2]:
!pip install mteb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 21.5 MB/s eta 0:00:00


In [3]:
def tokenize_chunking_strategy(tokenizer, inputs, chunk_size, overlap):
    real_chunks_size = chunk_size - tokenizer.num_special_tokens_to_add(pair=False)    # for each chunk special tokens will be appened after

    number_of_chunks = []    # number of chunks for each text in input
    outer_chunked_texts_batch = []    # long batch of all texts chunks

    for text in inputs:
        token_ids = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"].squeeze()
        start = 0
        chunk_number = 0
        while start < len(token_ids):
            end = start + real_chunks_size
            chunk_tokens = token_ids[start:end]
            chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            outer_chunked_texts_batch.append(chunk_text)
            start += real_chunks_size - overlap
            chunk_number += 1
        number_of_chunks.append(chunk_number)

    tokenized_outer_batch = tokenizer(
        outer_chunked_texts_batch,
        add_special_tokens=True,
        max_length=chunk_size,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    return (tokenized_outer_batch, number_of_chunks)

def re_group_chunked_outputs(outputs_last_hidden_state, attention_mask, number_of_chunks):
    # hidden_state shape: [ num_chunks * num_texts, chunk_size, hidden_size]
    # attention_mask shape: [ num_chunks * num_texts, chunk_size ]
    # converting to [num_texts, num_chunks, chunk_size, hidden_size]
    re_grouped_hidden_state = []
    re_grouped_attention_mask = []
    text_starts_i = 0
    for n_chunks in number_of_chunks:
        text_ends_i = text_starts_i + n_chunks
        re_grouped_hidden_state.append(
            outputs_last_hidden_state[text_starts_i:text_ends_i, :, :]
        )
        re_grouped_attention_mask.append(
            attention_mask[text_starts_i:text_ends_i, :]
        )
        text_starts_i = text_ends_i
    return re_grouped_hidden_state, re_grouped_attention_mask

def tokenize_first_startegy(tokenizer, inputs, max_length):
    tokenized = tokenizer(
        inputs,
        add_special_tokens=True,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    return tokenized


def tokenize_max_tokens_strategy(tokenizer, inputs):
    tokenized = tokenizer(
        inputs,
        add_special_tokens=True,
        padding="longest",
        truncation=True,
        max_length=tokenizer.model_max_length,
        return_tensors="pt")
    return tokenized

In [13]:
from mteb.similarity_functions import cos_sim

def get_hidden_states_per_batch(
        hidden_state,
        attention_mask,
        numbers_of_chunks,
        eos_token_only=True):
    """
    hidden_state - list: (batch_size, n_chunks (varying), chunk_size, hidden_size)
    attention_mask - list: (batch_size, n_chunks (varying), chunk_size)

    output should be a:
        - list of (batch_size, max_chunks * chunk_size, hidden_size) - tensor
        - new attention mask - tensor
    """

    def __eos_token_pool(hidden_state, attention_mask):
        # hidden_state (chunk_size, hidden_size)
        # attention_mask (chunk_size)
        sequence_lengths = attention_mask.sum(dim=0) - 1
        batch_size = hidden_state
        return hidden_state[sequence_lengths]

    batch_size = len(hidden_state)
    chunk_size = hidden_state[-1].shape[1]
    hidden_size = hidden_state[-1].shape[-1]

    device = hidden_state[-1].device
    dtype = hidden_state[-1].dtype

    max_n_chunks = max(numbers_of_chunks) if batch_size > 0 else 0
    max_len = chunk_size * max_n_chunks

    if eos_token_only:
        max_len = max_n_chunks

    hidden_states_padded = torch.zeros((batch_size, max_len, hidden_size), device=device, dtype=dtype)
    attn_mask_padded = torch.zeros((batch_size, max_len), device=device, dtype=attention_mask[-1].dtype)

    for t_i in range(batch_size):
        h_text = hidden_state[t_i]
        n_chunks = h_text.shape[0]
        if eos_token_only:
            L = n_chunks
            h_text = __eos_token_pool(h_text, attention_mask[t_i])
            print(h_text.shape)
            hidden_states_padded[t_i, :L, :] = h_text
            attn_mask_padded[t_i, :L] = 1
        else:
            h_text = h_text.reshape(chunk_size * n_chunks, hidden_size)
            mask_text = attention_mask[t_i]
            mask_text = mask_text.reshape(chunk_size * n_chunks)
            L = h_text.shape[0]
            hidden_states_padded[t_i, :L, :] = h_text
            attn_mask_padded[t_i, :L] = mask_text

    return hidden_states_padded, attn_mask_padded


class Memory:

    similarity = staticmethod(cos_sim)

    def __init__(
            self,
            batch_size,
            memory_size=5,
            hidden_size=1024,
            device="cuda",
            dtype=torch.float16):
        self.memory = torch.zeros(
            (batch_size, memory_size, hidden_size),
            device=device,
            dtype=dtype)
        self.memory_size = memory_size
        self.write_counts = torch.ones(
            (batch_size, memory_size),
            device=device,
            dtype=dtype)
        self.running_q = torch.zeros(
            (batch_size, hidden_size),
            device=device,
            dtype=dtype)

    def sample_h(self, hidden_state, attention_mask):
        for t_i in range(hidden_state.shape[0]):
            text_h = hidden_state[t_i]
            text_mask = attention_mask[t_i]
            text_valid_h = text_h[text_mask.bool()]
            # Handle cases where text_valid_h might be empty
            if text_valid_h.shape[0] == 0:
                continue
            sampled_idx = torch.randint(text_valid_h.shape[0], (self.memory_size, ))
            sampled_h = text_valid_h[sampled_idx]
            self.memory[t_i] = sampled_h
        return self.memory

    def write_memory(self, hidden_state, attention_mask):
        # hidden_state shape: [batch_size, max_chunks * chunk_size, hidden_size]
        # attention_mask shape: [batch_size, max_chunks * chunk_size]

        # first, fill the memory with random sampled hidden state vectors
        self.memory = self.sample_h(hidden_state, attention_mask)

        # next, iterate through the hidden_states and write the hidden state to the memory
        for i in range(hidden_state.shape[1]):
            h_all_batches = hidden_state[:, i, :]  # (batch_size, hidden_size)
            mask_all_batches = attention_mask[:, i]  # (batch_size)

            # only process active batches (where mask is 1)
            active_batches_idx = (mask_all_batches == 1).nonzero(as_tuple=True)[0]

            if len(active_batches_idx) == 0:
                continue # no active batches at this position, skip

            h = h_all_batches[active_batches_idx] # (num_active_batches, hidden_size)
            current_memory = self.memory[active_batches_idx] # (num_active_batches, memory_size, hidden_size)
            current_write_counts = self.write_counts[active_batches_idx] # (num_active_batches, memory_size)

            if i == 0:
                self.running_q[active_batches_idx] = h
            else:
                self.running_q[active_batches_idx] = self.running_q[active_batches_idx] * 0.9 + h * 0.1

            h_expanded = h.unsqueeze(1).expand_as(current_memory)
            sim = torch.cosine_similarity(h_expanded, current_memory, dim=2)  # (num_active_batches, memory_size)

            max_sim, max_sim_idx = torch.max(sim, dim=1)  # (num_active_batches)

            sim_prob = 1 - 1 / 2 * (1 - sim) # recalculate sim_prob for active batches

            # determine which active batches need forced rewrite (no similar memory found)
            needs_forced_rewrite = ~(sim_prob > 0.5).any(dim=1)

            # separate active batches into those that need regular update and those that need forced rewrite
            regular_update_mask = ~needs_forced_rewrite
            forced_rewrite_mask = needs_forced_rewrite

            regular_active_batches_idx = active_batches_idx[regular_update_mask]
            regular_max_sim_idx = max_sim_idx[regular_update_mask]
            regular_h = h[regular_update_mask]

            forced_active_batches_idx = active_batches_idx[forced_rewrite_mask]
            forced_current_write_counts = current_write_counts[forced_rewrite_mask]
            forced_h = h[forced_rewrite_mask]

            if len(forced_active_batches_idx) > 0:
                # rewrite the least occupied cell for these batches
                _, least_occupied_idx = torch.max(forced_current_write_counts, dim=1)
                self.memory[forced_active_batches_idx, least_occupied_idx] = forced_h
                self.write_counts[forced_active_batches_idx, least_occupied_idx] = 1 # reset count to 1 for new entry

            if len(regular_active_batches_idx) > 0:
                selected_regular = self.memory[regular_active_batches_idx, regular_max_sim_idx]
                w_counts_regular = self.write_counts[regular_active_batches_idx, regular_max_sim_idx]
                alpha_regular = 1 / w_counts_regular

                self.memory[regular_active_batches_idx, regular_max_sim_idx] = \
                    (1 - alpha_regular.unsqueeze(1)) * selected_regular + alpha_regular.unsqueeze(1) * regular_h
                self.write_counts[regular_active_batches_idx, regular_max_sim_idx] += 1

        return self.memory

    def pool_memory(self):
        # query shape: [batch_size, hidden_size]
        # memory shape: [batch_size, memory_size, hidden_size]

        # expand query to match memory dimensions for similarity calculation
        q_expanded = self.running_q.unsqueeze(1).expand_as(self.memory)

        # calculate cosine similarity between query and each memory slot
        # output shape: [batch_size, memory_size]
        sim = torch.cosine_similarity(q_expanded, self.memory, dim=2)

        # apply softmax to get attention weights
        # output shape: [batch_size, memory_size]
        attention_weights = torch.softmax(sim, dim=1)

        # expand attention weights to match hidden_size for weighted sum
        # output shape: [batch_size, memory_size, 1]
        attention_weights_expanded = attention_weights.unsqueeze(2)

        # perform weighted sum of memory states
        # output shape: [batch_size, hidden_size]
        pooled_memory = (attention_weights_expanded * self.memory).sum(dim=1)

        return pooled_memory

In [5]:
from mteb.types import PromptType

In [6]:
from mteb import EncoderProtocol

from enum import Enum

class Strategy(Enum):
    chunking = "chunking"
    first = "first"
    max_tokens = "max_tokens"
    memory = "memory"


class XMLRoBERTa(EncoderProtocol):

    name = "xlm-roberta-large"
    similarity = staticmethod(cos_sim)

    def __init__(self, max_size, strategy: Strategy, overlap):
        self.model = AutoModel.from_pretrained(
            "FacebookAI/xlm-roberta-large",
            dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map="auto")

        self.tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-large")

        self.max_size = max_size
        self.strategy = strategy
        self.overlap = overlap

        self.model.eval()

    def __encode_batch(
            self,
            texts,
            **kwargs) -> np.ndarray:

        if self.strategy == Strategy.chunking:
            tokenized, numbers_of_chunks = tokenize_chunking_strategy(self.tokenizer, texts, self.max_size, self.overlap)
        elif self.strategy == Strategy.first:
            tokenized = tokenize_first_startegy(self.tokenizer, texts, self.max_size)
        elif self.strategy == Strategy.max_tokens:
            tokenized = tokenize_max_tokens_strategy(self.tokenizer, texts)
        else:
            raise ValueError(f"Unknown strategy: {self.strategy}")

        with torch.inference_mode():
            tokenized = {k: v.to(self.model.device) for k, v in tokenized.items()}
            attention_mask = tokenized["attention_mask"]
            outputs = self.model(**tokenized)

            if self.strategy == Strategy.chunking:
                re_grouped_hidden_state, re_grouped_attention_mask = re_group_chunked_outputs(outputs.last_hidden_state, attention_mask, numbers_of_chunks)

                list_of_text_embeddings = []
                for i in range(len(re_grouped_hidden_state)):
                    hidden_states_for_text = re_grouped_hidden_state[i] # [num_chunks, chunk_size, hidden_size]
                    attention_mask_for_text = re_grouped_attention_mask[i] # [num_chunks, chunk_size]

                    masked_hidden_states = hidden_states_for_text * attention_mask_for_text.unsqueeze(-1)
                    sum_embeddings_per_chunk = masked_hidden_states.sum(dim=1)
                    num_valid_tokens_per_chunk = attention_mask_for_text.sum(dim=1)
                    num_valid_tokens_per_chunk_clamped = torch.clamp(num_valid_tokens_per_chunk, min=1)

                    # Weighted average over chunk_size dimension for each chunk
                    chunk_embeddings = sum_embeddings_per_chunk / num_valid_tokens_per_chunk_clamped.unsqueeze(-1)
                    text_embedding = chunk_embeddings.mean(dim=0)
                    list_of_text_embeddings.append(text_embedding)
                embeddings = torch.stack(list_of_text_embeddings)

            elif (self.strategy == Strategy.first
                or self.strategy == Strategy.max_tokens):
                # Apply attention mask for mean pooling
                masked_output = outputs.last_hidden_state * attention_mask.unsqueeze(-1)
                sum_embeddings = masked_output.sum(dim=1)
                num_valid_tokens = attention_mask.sum(dim=1)
                num_valid_tokens_clamped = torch.clamp(num_valid_tokens, min=1)
                embeddings = sum_embeddings / num_valid_tokens_clamped.unsqueeze(-1)
            else:
                raise ValueError(f"Unhandled strategy for embedding calculation: {self.strategy}")

        embeddings = embeddings.detach().cpu().numpy()

        del outputs, tokenized
        torch.cuda.empty_cache()
        return embeddings

    def encode(
            self,
            inputs,
            *,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type,
            **kwargs):

        batch_size = kwargs["batch_size"]
        return_hidden_state = kwargs["return_hidden_state"]

        texts = [text for batch in inputs for text in batch["text"]]

        all_embeddings = []

        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            batch_embeddings = self.__encode_batch(batch, return_hidden_state)
            all_embeddings.append(batch_embeddings)

        return_embeddings = np.vstack(all_embeddings)
        print(return_embeddings.shape)
        return return_embeddings



class Qwen3_Embedding(EncoderProtocol):

    name = "Qwen3-Embedding-0.6B"
    similarity = staticmethod(cos_sim)

    def __init__(
            self,
            max_size,
            strategy,
            overlap,
            return_hidden_states=False,
            **kwargs):
        if strategy == Strategy.memory:
            self.tokenizer = AutoTokenizer.from_pretrained(
                "Qwen/Qwen3-Embedding-0.6B",
                padding_side='right')
            self.memory_size = kwargs["memory_size"]
        else:
            self.tokenizer = AutoTokenizer.from_pretrained(
                "Qwen/Qwen3-Embedding-0.6B",
                padding_side='left')

        self.model = AutoModel.from_pretrained(
            "Qwen/Qwen3-Embedding-0.6B",
            dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map="auto")

        self.max_size = max_size
        self.strategy = strategy
        self.overlap = overlap

        self.return_hidden_states = return_hidden_states

        self.model.eval()

    def __get_eos_token_embedding(self, last_hidden_states):
        return last_hidden_states[:, -1]

    def preprocess_query(self, texts):
        task = 'Given a search query, retrieve relevant passages that answer the query'
        return f'Instruct: {task}\nQuery:{texts}'

    @torch.no_grad()
    def __encode_batch(self, texts, **kwargs) -> np.ndarray:
        return_hidden_state = kwargs["return_hidden_state"]
        if self.strategy == Strategy.chunking:
            tokenized, numbers_of_chunks = tokenize_chunking_strategy(
                self.tokenizer, texts, self.max_size, self.overlap
            )
        elif self.strategy == Strategy.first:
            tokenized = tokenize_first_startegy(self.tokenizer, texts, self.max_size)
        elif self.strategy == Strategy.memory:
            memory = Memory(
                len(texts),
                memory_size=self.memory_size,
                hidden_size=1024,
                device=self.model.device,
                dtype=self.model.dtype)
            tokenized, numbers_of_chunks = tokenize_chunking_strategy(
                self.tokenizer, texts, self.max_size, self.overlap
            )

        with torch.inference_mode():
            tokenized = {k: v.to(self.model.device) for k, v in tokenized.items()}
            attention_mask = tokenized["attention_mask"]
            outputs = self.model(**tokenized)

            if self.strategy == Strategy.chunking or self.strategy == Strategy.memory:
                re_grouped_hidden_state, re_grouped_attention_mask = re_group_chunked_outputs(
                    outputs.last_hidden_state, attention_mask, numbers_of_chunks
                )
                if return_hidden_state:

                    del outputs, tokenized
                    torch.cuda.empty_cache()

                    return {
                        "hidden_state": [
                            h.detach().cpu().numpy()
                            for h in re_grouped_hidden_state
                        ],
                        "attention_mask": [
                            a.detach().cpu().numpy()
                            for a in re_grouped_attention_mask
                        ]
                    }

            if self.strategy == Strategy.chunking:
                embeddings = torch.stack([
                    self.__get_eos_token_embedding(t).mean(dim=0)
                    for t in re_grouped_hidden_state
                ])
            if self.strategy == Strategy.memory:
                hidden_states_padded, attention_padded = get_hidden_states_per_batch(
                    re_grouped_hidden_state, re_grouped_attention_mask, numbers_of_chunks
                )
                memory.write_memory(hidden_states_padded, attention_padded)
                embeddings = memory.pool_memory()
            else:
                embeddings = self.__get_eos_token_embedding(
                    outputs.last_hidden_state
                )

        embeddings = embeddings.detach().cpu().numpy()

        del outputs, tokenized
        if self.strategy == Strategy.memory:
            del hidden_states_padded, attention_padded, memory
        torch.cuda.empty_cache()
        return embeddings

    def encode(
            self,
            inputs,
            *,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type,
            **kwargs):

        batch_size = kwargs["batch_size"]
        return_hidden_state = kwargs["return_hidden_state"]

        texts = [text for batch in inputs for text in batch["text"]]
        if prompt_type.value == "query":
            texts = [self.preprocess_query(t) for t in texts]

        all_embeddings = []

        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            batch_embeddings = self.__encode_batch(
                batch,
                return_hidden_state=return_hidden_state)
            all_embeddings.append(batch_embeddings)
            if return_hidden_state:
                return all_embeddings

        return_embeddings = np.vstack(all_embeddings)
        print("Return Size")
        print(return_embeddings.shape)
        return return_embeddings

In [7]:
import seaborn as sns
def plot_hidden_state_similarity(model, texts):
    outputs = model.encode(
        texts,
        task_metadata=None,
        hf_split=None,
        hf_subset=None,
        prompt_type=PromptType.document,
        batch_size=1,
        return_hidden_state=True)

    embedding = model.encode(
        texts,
        task_metadata=None,
        hf_split=None,
        hf_subset=None,
        prompt_type=PromptType.document,
        batch_size=1,
        return_hidden_state=False)

    hidden_state = []
    attention_mask = []
    for chunk in outputs[0]["hidden_state"]:
        hidden_state.extend(chunk)
    hidden_state = np.vstack(hidden_state)

    for chunk in outputs[0]["attention_mask"]:
        attention_mask.extend(chunk)
    attention_mask = np.array(attention_mask)
    attention_mask = attention_mask.reshape(1, -1)[0]

    hidden_state = hidden_state[attention_mask.astype(bool)]

    emb_h_similarity = cos_sim(embedding, hidden_state)[0]

    plt.figure(figsize=(12, 6))

    # Plot token similarities using seaborn
    sns.lineplot(x=list(range(len(emb_h_similarity))), y=emb_h_similarity.numpy(), label='Similarity per token')

    # Add chunk borders
    for border in chunk_borders:
        plt.axvline(x=border, color='r', linestyle='--', label='Chunk Border' if border == chunk_borders[0] else "")

    # Calculate mean and variance for the legend
    mean_similarity = emb_h_similarity.mean().item() # .item() to get scalar from tensor
    var_similarity = emb_h_similarity.var().item() # .item() to get scalar from tensor

    # Add horizontal line for mean similarity with updated legend
    plt.axhline(y=mean_similarity, color='g', linestyle='--', label=f"Mean Similarity ($\mu$ = {mean_similarity:.2f}, $\sigma^2$ = {var_similarity:.2f})")

    plt.xlabel('Token Index')
    plt.ylabel('Cosine Similarity')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.show()

<>:50: SyntaxWarning: invalid escape sequence '\m'
<>:50: SyntaxWarning: invalid escape sequence '\s'
<>:50: SyntaxWarning: invalid escape sequence '\m'
<>:50: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipython-input-3431551190.py:50: SyntaxWarning: invalid escape sequence '\m'
  plt.axhline(y=mean_similarity, color='g', linestyle='--', label=f"Mean Similarity ($\mu$ = {mean_similarity:.2f}, $\sigma^2$ = {var_similarity:.2f})")
/tmp/ipython-input-3431551190.py:50: SyntaxWarning: invalid escape sequence '\s'
  plt.axhline(y=mean_similarity, color='g', linestyle='--', label=f"Mean Similarity ($\mu$ = {mean_similarity:.2f}, $\sigma^2$ = {var_similarity:.2f})")


# Evaluation

In [8]:
import json
import mteb


In [9]:
qwen3_memory = Qwen3_Embedding(
    max_size=512,
    strategy=Strategy.memory,
    overlap=0,
    memory_size=6)

qwen3_chunking = Qwen3_Embedding(
    max_size=512,
    strategy=Strategy.chunking,
    overlap=0)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

In [10]:
 long_embed_wiki_task = mteb.get_task("LEMBWikimQARetrieval")

In [14]:
def evaluate(
    model,
    task):
    print("------- Evaluation of model --------")
    print(f"    Transformer: {model.name}")
    print(f"    Strategy: {model.strategy.value}")
    eval_results = task.evaluate(model, encode_kwargs={"batch_size": 20, "return_hidden_state": False})
    file_name = f"{task.metadata.name}_{model.name}_{model.strategy.value}.json"
    with open(file_name, "w") as f:
        json.dump(eval_results, f)
    print(f"     Main score: {eval_results['default']['main_score']:.4f}")
    return eval_result

evaluate(
    qwen3_memory,
    task=long_embed_wiki_task
)

evaluate(
    qwen3_chunking,
    task=long_embed_wiki_task
)


------- Evaluation of model --------
    Transformer: Qwen3-Embedding-0.6B
    Strategy: memory


Filtering queries by qrels:   0%|          | 0/300 [00:00<?, ? examples/s]

Processing queries for dataloading:   0%|          | 0/300 [00:00<?, ? examples/s]

RuntimeError: expand(torch.cuda.HalfTensor{[512, 512, 1024]}, size=[1, 1024]): the number of sizes provided (2) must be greater or equal to the number of dimensions in the tensor (3)

In [ ]:
/